# 📊 Polymarket Trader Explorer — EDA

**2.383 traders** coletados de 9 categorias × 4 períodos

**Variáveis disponíveis:**
- `pnl` — Lucro total
- `volume` — Volume total negociado
- `efficiency` — PnL / Volume (qualidade das decisões)
- `n_appearances` — Quantas vezes aparece nos leaderboards
- `n_periods` — Em quantos timeframes aparece (1-4: ALL/MONTH/WEEK/DAY)
- `n_categories` — Em quantas categorias opera
- `categories` — Lista de categorias (POLITICS, CRYPTO, SPORTS, etc)
- `in_day/in_week/in_month/in_all` — Presença em cada período
- `best_overall_rank` — Melhor posição no ranking geral

In [ ]:
import pandas as pd
import numpy as np
import json

# Carrega dados
df = pd.read_json('eda_broad.json')
print(f'Total traders: {len(df)}')
df.head()

In [ ]:
# Overview das variáveis
df[['pnl', 'volume', 'efficiency', 'n_appearances', 'n_periods', 'n_categories']].describe()

In [ ]:
# Distribuição de PnL
bins = [0, 1e3, 1e4, 1e5, 5e5, 1e6, 5e6, 1e7, 1e8]
labels = ['<$1K', '$1K-10K', '$10K-100K', '$100K-500K', '$500K-1M', '$1M-5M', '$5M-10M', '$10M+']
df['pnl_bucket'] = pd.cut(df['pnl'], bins=bins, labels=labels)
df['pnl_bucket'].value_counts().sort_index()

In [ ]:
# Distribuição de Eficiência
eff_bins = [0, 0.01, 0.05, 0.1, 0.2, 0.5, 1.0, 10]
eff_labels = ['<1%', '1-5%', '5-10%', '10-20%', '20-50%', '50-100%', '>100%']
df['eff_bucket'] = pd.cut(df['efficiency'], bins=eff_bins, labels=eff_labels)
df['eff_bucket'].value_counts().sort_index()

In [ ]:
# Consistência: quantos traders aparecem em múltiplos períodos?
print('Distribuição por nº de períodos:')
print(df['n_periods'].value_counts().sort_index())
print(f'\nTraders em 3+ períodos (consistentes): {(df.n_periods >= 3).sum()}')
print(f'Traders no leaderboard DIÁRIO (ativos agora): {df.in_day.sum()}')

In [ ]:
# Categorias: quais são mais populares?
from collections import Counter
all_cats = [c for cats in df['categories'] for c in cats]
cat_counts = Counter(all_cats)
pd.Series(cat_counts).sort_values(ascending=False)

In [ ]:
# Especialistas vs Generalistas
specialists = df[df.n_categories == 1]
generalists = df[df.n_categories >= 3]

print(f'Especialistas (1 cat): {len(specialists)} — Efic média: {specialists.efficiency.mean()*100:.1f}%')
print(f'Generalistas (3+ cat): {len(generalists)} — Efic média: {generalists.efficiency.mean()*100:.1f}%')
print(f'\nEspecialistas por categoria:')
for cat in ['POLITICS', 'CRYPTO', 'SPORTS', 'ECONOMICS', 'FINANCE']:
    subset = specialists[specialists.categories.apply(lambda x: cat in x)]
    if len(subset) > 0:
        print(f'  {cat}: {len(subset)} traders — PnL médio: ${subset.pnl.mean():,.0f} — Efic: {subset.efficiency.mean()*100:.1f}%')

In [ ]:
# Scatter: PnL vs Eficiência (colorido por nº de períodos)
try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(12, 6))
    scatter = ax.scatter(
        df['efficiency'] * 100, df['pnl'] / 1e6,
        c=df['n_periods'], cmap='RdYlGn', s=df['n_appearances']*3,
        alpha=0.6, edgecolors='white', linewidth=0.3
    )
    plt.colorbar(scatter, label='Nº Períodos (consistência)')
    ax.set_xlabel('Eficiência (%)')
    ax.set_ylabel('PnL ($M)')
    ax.set_title('PnL vs Eficiência — Tamanho = Aparições, Cor = Consistência')
    # Label top traders
    for _, row in df.nlargest(10, 'pnl').iterrows():
        ax.annotate(row['name'] or row['wallet'][:8], (row['efficiency']*100, row['pnl']/1e6), fontsize=7)
    plt.tight_layout()
    plt.show()
except ImportError:
    print('matplotlib não instalado — instale com: pip install matplotlib')

In [ ]:
# Correlação entre variáveis
df[['pnl', 'volume', 'efficiency', 'n_appearances', 'n_periods', 'n_categories']].corr().round(2)

## 🎯 Modelagem de Seleção

Use as células abaixo para testar diferentes critérios de seleção e ver quem cai no filtro.

In [ ]:
# === AJUSTE SEUS CRITÉRIOS AQUI ===

selected = df[
    (df.pnl >= 100_000) &           # PnL mínimo
    (df.efficiency >= 0.05) &         # Eficiência mínima (5%)
    (df.n_periods >= 2) &             # Aparece em 2+ períodos
    (df.n_categories >= 1)            # Pelo menos 1 categoria
]

print(f'Selecionados: {len(selected)} de {len(df)} traders')
print(f'PnL médio: ${selected.pnl.mean():,.0f}')
print(f'Eficiência média: {selected.efficiency.mean()*100:.1f}%')
print(f'\nTop 20:')
selected.nlargest(20, 'pnl')[['name', 'pnl', 'volume', 'efficiency', 'n_periods', 'n_categories', 'categories']].to_string()

In [ ]:
# Seleção por categoria específica
CATEGORY = 'CRYPTO'  # Mude aqui: POLITICS, CRYPTO, SPORTS, ECONOMICS, FINANCE

cat_df = df[df.categories.apply(lambda x: CATEGORY in x)].copy()
cat_df = cat_df.sort_values('pnl', ascending=False)

print(f'\n=== {CATEGORY} — {len(cat_df)} traders ===')
print(cat_df.head(15)[['name', 'pnl', 'volume', 'efficiency', 'n_periods', 'n_categories']].to_string())

In [ ]:
# Score composto — customize os pesos
def compute_score(row, w_pnl=0.3, w_eff=0.25, w_consist=0.25, w_spec=0.2):
    pnl_norm = min(row['pnl'] / 5_000_000, 1.0)
    eff_norm = min(row['efficiency'] / 0.5, 1.0)
    consist_norm = row['n_periods'] / 4
    spec_norm = min(row['n_appearances'] / 15, 1.0)
    return round(pnl_norm * w_pnl + eff_norm * w_eff + consist_norm * w_consist + spec_norm * w_spec, 4)

df['score'] = df.apply(compute_score, axis=1)
print('Top 20 por Score Composto:')
df.nlargest(20, 'score')[['name', 'score', 'pnl', 'efficiency', 'n_periods', 'n_categories', 'categories']].to_string()

In [ ]:
# === COLETA EXTRA: Posições atuais dos selecionados ===
# Rode esta célula para buscar posições em tempo real dos traders que você escolheu

import requests, time

DATA_API = 'https://data-api.polymarket.com'

# Pega os top N pelo seu critério
TOP_N = 10
targets = selected.nlargest(TOP_N, 'pnl')

for _, t in targets.iterrows():
    w = t['wallet']
    name = t['name'] or w[:12]
    r = requests.get(f'{DATA_API}/positions', params={'user': w, 'sizeThreshold': '0'})
    pos = r.json() if isinstance(r.json(), list) else []
    print(f'\n{name} — {len(pos)} posições:')
    for p in pos[:5]:
        print(f'  {p.get("title","")[:50]} → {p.get("outcome")} (${float(p.get("size",0)):,.0f})')
    time.sleep(0.5)